# Sanctions Screening: AI Similarity Matching

Uses Databricks **`ai_similarity()`** (semantic / embedding-based similarity) to compare holdings against sanctioned entities, then compares results with the character-based `thefuzz` scores from notebook 02.

### Approaches in this notebook

| Method | What it does | Best for |
|--------|-------------|----------|
| `ai_similarity()` | Embedding-based semantic score (0–1) | Catching meaning-level matches; filtering false positives from fuzzy matching |
| `ai_query()` (fallback) | LLM yes/no classification | Transliteration, abbreviation, and context-aware matching when embeddings fall short |

> **Note:** `ai_similarity` requires a workspace region that supports [AI Functions](https://docs.databricks.com/aws/en/resources/feature-region-support#ai-aws) and is not available on SQL Classic.

In [ ]:
CATALOG      = "renjiharold_demo"
SCHEMA       = "sanctions_screening"
SOURCE_TABLE = f"{CATALOG}.{SCHEMA}.holdings_sanctioned_pairs"
RESULT_TABLE = f"{CATALOG}.{SCHEMA}.ai_similarity_results"

spark.sql(f"USE CATALOG {CATALOG}")
spark.sql(f"USE SCHEMA {SCHEMA}")

n_rows = spark.table(SOURCE_TABLE).count()
print(f"Source table: {SOURCE_TABLE}  ({n_rows:,} rows)")

## 1. Run `ai_similarity()` on all 10,000 pairs

The function is a Spark SQL built-in — it batches calls to the Foundation Model endpoint automatically.  
Expect this cell to take several minutes for 10K rows.

In [ ]:
import time

start = time.time()
spark.sql(f"""
    CREATE OR REPLACE TABLE {RESULT_TABLE} AS
    SELECT
        holding,
        sanctioned,
        ai_similarity(holding, sanctioned) AS ai_similarity_score
    FROM {SOURCE_TABLE}
""")
elapsed = time.time() - start
print(f"ai_similarity completed in {elapsed:.1f}s for {n_rows:,} rows")

In [ ]:
display(
    spark.sql(f"""
        SELECT holding, sanctioned, ai_similarity_score
        FROM {RESULT_TABLE}
        ORDER BY ai_similarity_score DESC
        LIMIT 50
    """)
)

## 2. Compare `ai_similarity` with `thefuzz` scores

Joins the AI similarity results with the thefuzz results (from notebook 02) to see where the two methods agree and disagree.  
High `fuzz_token_set_ratio` + low `ai_similarity_score` → likely **false positive** in fuzzy matching.

In [ ]:
thefuzz_table = f"{CATALOG}.{SCHEMA}.thefuzz_results"

if spark.catalog.tableExists(thefuzz_table):
    comparison_df = spark.sql(f"""
        SELECT
            a.holding,
            a.sanctioned,
            t.fuzz_ratio,
            t.fuzz_token_set_ratio,
            ROUND(a.ai_similarity_score, 4) AS ai_sim_score,
            CASE
                WHEN a.ai_similarity_score >= 0.7 THEN 'HIGH'
                WHEN a.ai_similarity_score >= 0.5 THEN 'MEDIUM'
                ELSE 'LOW'
            END AS ai_match_level
        FROM {RESULT_TABLE} a
        JOIN {thefuzz_table} t
          ON a.holding = t.holding AND a.sanctioned = t.sanctioned
        WHERE t.fuzz_token_set_ratio >= 50 OR a.ai_similarity_score >= 0.5
        ORDER BY a.ai_similarity_score DESC
    """)
    display(comparison_df)
else:
    print(f"Table {thefuzz_table} not found — run notebook 02 first for side-by-side comparison.")

### Score distribution comparison

In [ ]:
if spark.catalog.tableExists(thefuzz_table):
    display(
        spark.sql(f"""
            SELECT
                CASE
                    WHEN t.fuzz_token_set_ratio >= 80 AND a.ai_similarity_score >= 0.7 THEN 'Both HIGH — likely match'
                    WHEN t.fuzz_token_set_ratio >= 60 AND a.ai_similarity_score <  0.5 THEN 'Fuzzy HIGH / AI LOW — false positive'
                    WHEN t.fuzz_token_set_ratio <  40 AND a.ai_similarity_score >= 0.7 THEN 'Fuzzy LOW / AI HIGH — fuzzy missed it'
                    ELSE 'Both LOW — no match'
                END AS verdict,
                count(*) AS pair_count
            FROM {RESULT_TABLE} a
            JOIN {thefuzz_table} t
              ON a.holding = t.holding AND a.sanctioned = t.sanctioned
            GROUP BY 1
            ORDER BY pair_count DESC
        """)
    )

## 3. Alternative — `ai_query()` for LLM-based entity resolution

If `ai_similarity` gives low scores on known matches (e.g. transliterations like *Gazprom Holdings* ↔ *Gazprom OOO*), an LLM can reason about abbreviations, transliterations, and legal suffixes.

**Trade-off:** slower and more expensive — run only on a **filtered subset** (e.g. pairs where `fuzz_token_set_ratio >= 60`).

This is the recommended production pipeline:
1. **First pass** — `thefuzz` (fast, cheap) → keep pairs above threshold
2. **Second pass** — `ai_similarity` or `ai_query` (accurate, costly) → only on candidates from step 1

In [ ]:
AI_QUERY_TABLE = f"{CATALOG}.{SCHEMA}.ai_query_results"

if spark.catalog.tableExists(thefuzz_table):
    spark.sql(f"""
        CREATE OR REPLACE TABLE {AI_QUERY_TABLE} AS
        WITH candidates AS (
            SELECT holding, sanctioned, fuzz_ratio, fuzz_token_set_ratio
            FROM {thefuzz_table}
            WHERE fuzz_token_set_ratio >= 60
        )
        SELECT
            c.holding,
            c.sanctioned,
            c.fuzz_ratio,
            c.fuzz_token_set_ratio,
            ai_similarity(c.holding, c.sanctioned) AS ai_sim_score,
            ai_query(
                'databricks-meta-llama-3-3-70b-instruct',
                CONCAT(
                    'You are a compliance analyst. Determine if these two entity names refer to the SAME company or organisation. ',
                    'Consider transliterations, abbreviations, acronyms, and legal suffixes ',
                    '(OOO/ZAO/AO = Russian entity types; LLC/Inc/Ltd/Corp = English). ',
                    'Answer ONLY with YES or NO.\n',
                    'Name 1: ', c.holding, '\n',
                    'Name 2: ', c.sanctioned
                )
            ) AS ai_verdict
        FROM candidates c
    """)
    display(spark.table(AI_QUERY_TABLE).orderBy("fuzz_token_set_ratio", ascending=False))
else:
    print(f"Run notebook 02 first to generate {thefuzz_table}.")

## Summary & recommendations

| Method | Speed (10K rows) | Strengths | Weaknesses |
|--------|-----------------|-----------|------------|
| **thefuzz** (Spark UDF) | Seconds | Fast; catches typos and character edits | High false-positive rate on shared words (e.g. "Holdings") |
| **`ai_similarity()`** | Minutes | Semantic meaning; reduces false positives | May miss transliterations; embedding-level, not reasoning |
| **`ai_query()`** | Minutes–hours | Full reasoning; handles transliteration + context | Slowest and most expensive |

### Recommended pipeline for production

```
10,000 pairs
  │
  ├─ Step 1: thefuzz token_set_ratio (Spark UDF)  →  keep score ≥ 60  →  ~200 candidates
  │
  ├─ Step 2: ai_similarity on candidates           →  keep score ≥ 0.5 →  ~50 pairs
  │
  └─ Step 3: ai_query YES/NO on remaining          →  final verdict    →  ~10 confirmed matches
```

This **funnel approach** keeps cost and latency low while maximising accuracy.